In [1]:
import warnings
warnings.filterwarnings('ignore')

(Tutorial_Get_lipophilicity)=
# Get lipophilicity

*Calculating state-preserving molecular Wildman–Crippen model scores.*

The experimental {func}`molsysmt.physchem.get_lipophilicity` exposes the atomic
contribution model of [Wildman and Crippen (1999)](https://doi.org/10.1021/ci990307l)
through RDKit. Each complete molecule gives one dimensionless score. This can be
used as a calculated logP estimate in an appropriate uncharged-species domain;
it is not a pH-dependent logD prediction. No chemical preparation is performed.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

See {func}`molsysmt.physchem.get_lipophilicity` for arguments, results and errors.
:::


## Basic usage

Let's show how this function works with two complete chemical graphs. RDKit is
an optional provider; any supported input form with the necessary chemistry,
hydrogen inventory and molecule membership can be used. No coordinates are needed.


In [2]:
import molsysmt as msm
from rdkit import Chem

molsys = msm.convert(Chem.MolFromSmiles('CCO.c1ccccc1'), to_form='molsysmt.MolSys')
scores = msm.physchem.get_lipophilicity(molsys, method='wildman_crippen')
scores.round(4).tolist()


[-0.0014, 1.6866]

## Selecting complete molecules

The default `element='molecule'` selects source molecule indices, preserving
order and repetitions. Other element levels address their own indices. A string selection uses
`syntax` to identify whole parent molecules, rather than capped atom fragments.
Independent molecules give separate values, with no whole-system sum.


In [3]:
result = msm.physchem.get_lipophilicity(molsys, selection=[1, 0, 1], return_report=True)
selected = msm.physchem.get_lipophilicity(molsys, selection='atom_index==0')
empty = msm.physchem.get_lipophilicity(molsys, selection=[])
print(result['report']['molecule_indices'].tolist())
print(result['report']['atom_offsets'].tolist())
print(selected.round(4).tolist(), empty.shape, empty.dtype)


[1, 0, 1]
[0, 6, 9, 15]
[-0.0014] (0,) float64


## Preserving the chemical state

`chemical_state='reference'` uses the stored reference; an integer chooses a
stored state. With `chemical_state='structure'`, `structure_indices` must resolve
to one chemical state. Different associated states are not averaged. Spatial
selection may need coordinates, but the descriptor has no structures output axis.

The following separately supplied acid and conjugate-base graphs produce different
model scores. Neither is automatically changed into the other.


In [4]:
molsys = msm.convert(Chem.MolFromSmiles('CC(=O)O.CC(=O)[O-]'), to_form='molsysmt.MolSys')
result = msm.physchem.get_lipophilicity(molsys, chemical_state='reference', return_report=True)
print(result['lipophilicity'].round(4).tolist())
print(result['report']['formal_charges'].tolist())


[0.0909, -1.2438]
[0, -1]


## Interpreting charged sites

Both nitro groups and zwitterions can have formal atomic charges but net charge
zero. The report exposes those charged source atoms without pretending that a
charge test can distinguish their physical applicability. Scores for supported
ions and zwitterions are model outputs, not validated physical partition coefficients.

:::{warning}
Do not interpret these results as logD at a specified pH. No pKa, species
populations, neutralization, counterion removal or tautomer selection is computed.
:::


In [5]:
molsys = msm.convert(Chem.MolFromSmiles('C[N+](=O)[O-].[NH3+]CC(=O)[O-]'), to_form='molsysmt.MolSys')
result = msm.physchem.get_lipophilicity(molsys, return_report=True)
report = result['report']
print(result['lipophilicity'].round(4).tolist())
print(report['formal_charges'].tolist())
print(report['charged_atom_indices'].tolist(), report['charged_atom_offsets'].tolist())


[-0.1071, -3.0218]
[0, 0]
[1, 3, 4, 8] [0, 2, 4]


## Keeping dimensionless scores

The return value is a float64 NumPy array, not an area or an energy quantity.
Changing the application's PyUnitWizard length policy does not rescale a
logarithmic model score. Negative scores are valid model outputs.


In [6]:
with msm.pyunitwizard.context(standard_units=['pm', 'ps', 'radians']):
    changed_policy = msm.physchem.get_lipophilicity(molsys)
print(changed_policy.round(4).tolist())
print(changed_policy.dtype, changed_policy.shape)


[-0.1071, -3.0218]
float64 (2,)


## Retaining provenance and correspondence

`return_report=True` pairs `lipophilicity` with a detached `report`. Flat source
atom indices and offsets align with the selected molecule order; charged atom
indices have their own offsets. The report records the state/digest, dimensionless
value kind, method, hydrogen treatment, chemical preparation policy, original
producer versions and bibliography. It distinguishes the original model from
the executed RDKit implementation. Optional Ackredit contributes to the caller's
current session. Reading exported metadata must not claim a new calculation.


In [7]:
print(report['schema'], report['value_kind'], report['value_unit'])
print(report['method'], report['parameters'])
print(report['software'])
print([item['id'] for item in report['attribution']['items']])


molsysmt.lipophilicity@1 wildman_crippen_model_score dimensionless
wildman_crippen {'hydrogen_treatment': 'preserve_declared_inventory', 'implementation': 'rdkit.CalcCrippenDescriptors', 'include_hydrogen_contributions': True, 'chemical_preparation': 'none'}
{'molsysmt': '0.23.0+201.g3502232d8.dirty', 'rdkit': '2025.09.5'}
['rdkit-aromaticity-model', 'software:molsysmt:0.23.0+201.g3502232d8.dirty', 'software:rdkit:2025.09.5', 'doi:10.1021/ci990307l']


## Selecting atomic contributions

Use `element='atom'` to select atom indices. Contributions are evaluated on the
complete parent molecule, including atoms outside the output selection. They
are not descriptors of isolated capped fragments. Numeric order and repetitions
are retained, and each needed parent molecule is evaluated once.


In [8]:
molsys = msm.convert(Chem.MolFromSmiles('CCO'), to_form='molsysmt.MolSys')
atomic = msm.physchem.get_lipophilicity(molsys, element='atom', selection=[2, 0, 2])
atomic.round(4).tolist()


[-0.557, 0.5131, -0.557]

## Grouping the contributions

The same pattern accepts `group`, `component`, `chain` and `entity`. Numeric
selection addresses that element level; text selects whole elements. Here we
explicitly define two groups in one molecule to show that grouping does not
change the chemical graph or require cutting its bonds. The polar group contains
atoms 1 and 2, and the alkyl group contains atom 0. These are illustrative groups,
not a rule for partitioning ligands.


In [9]:
builder = msm.build.editable(molsys)
builder.add_group([1, 2], group_name='POLAR', group_type='small molecule')
builder.add_group([0], group_name='ALKYL', group_type='small molecule')
molsys = builder.build()
grouped = msm.physchem.get_lipophilicity(molsys, element='group', selection=[1, 0, 1])
grouped.round(4).tolist()


[0.5131, -0.5145, 0.5131]

## Summing a selection

Use `element='system'` for a scalar sum over selected atoms. Numeric selections
are atom indices in this mode. Repeated indices repeat their contribution; an
empty atom selection gives zero. At other element levels an empty selection
returns shape `(0,)`. Sums spanning molecules are additive model contributions,
not a physical mixture partition coefficient or exposed surface measurement.


In [10]:
total = msm.physchem.get_lipophilicity(molsys, element='system', selection=[0, 2])
float(total)


-0.04389999999999994

## Interpreting hydrogen attribution and source maps

A hydrogen with a source atom index keeps its own contribution. A nonindexed
hydrogen contributes to its retained attachment atom. Therefore individual heavy
atom contributions can differ between explicit and implicit H representations;
folding indexed H contributions onto their attachment atoms restores agreement.
TPSA assigns zero to H atoms; their chemical context still determines polar-atom
contributions. No source atoms are added or removed by either calculation.

In a report, `element_indices`, `element_atom_indices` and
`element_atom_offsets` identify output rows. `molecule_indices`, `atom_indices`
and `atom_offsets` identify the complete molecules evaluated as context. Their
row counts can differ. `output_kind` identifies contribution sums versus complete
molecular descriptors. Charge and charged-site arrays refer to these evaluated
molecular contexts. Keep the distinction when exporting a subset.


In [11]:
result = msm.physchem.get_lipophilicity(molsys, element='atom', selection=[2, 0, 2], return_report=True)
report = result['report']
print(report['element'], report['element_indices'].tolist())
print(report['element_atom_indices'].tolist(), report['element_atom_offsets'].tolist())
print(report['molecule_indices'].tolist(), report['atom_indices'].tolist())
print(report['output_kind'])


atom [2, 0, 2]
[2, 0, 2] [0, 1, 2, 3]
[0] [0, 1, 2]
atomic_contribution_sum


## Understanding the limits

Hydrogen counts, formal charges, connectivity and a supported closed-shell graph
must be declared. Provider materialization of already declared hydrogens happens
on a temporary graph. Unspecified native SDF hydrogen inventory fails clearly.
No source coordinates, atom assignments or provider caches are changed.

Disconnected salt labels, molecule boundaries crossed by covalent bonds, metals,
radicals, query/dummy atoms and dative relationships fail in this first profile.
A connected ion is different from a disconnected salt label. A computable score
does not certify a calibrated prediction domain. Independent primary atomic-sum
controls cover methane, ethane and benzene; broader tests establish contract and
provider parity, not general experimental accuracy. No new Rust kernel, descriptor
persistence format or automatic MolSys/H5MSM attachment is introduced.

:::{seealso}
:class: dropdown

- {func}`molsysmt.physchem.get_lipophilicity` for the complete descriptor contract.
- {func}`molsysmt.physchem.get_tpsa` for a distinct, unit-aware polar surface descriptor.
- {ref}`Cookbook_Molecular_Descriptors` for joining values to source maps.
:::
